# 03 — Change Score Prediction (Q3)

**Research Question:** Does the change in dopaminergic brain structure predict the change in PQ-BC severity?

Target: Delta PPS = Year 2 PPS - Baseline PPS
Features: Delta Brain = Year 2 Brain - Baseline Brain

Uses `merge_longitudinal(imaging_cols=...)` to compute both delta target and delta brain features.

**ICV correction is disabled** for delta analyses (ratio correction doesn't apply to difference scores).

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from copy import deepcopy

from src.core.config import initialize_notebook
env = initialize_notebook(run_name='regression')

## 1. Build longitudinal dataset

In [ ]:
from src.core.preprocessing.ingest import load_and_merge
from src.core.preprocessing.splits import timepoint_split, merge_longitudinal
from src.core.preprocessing.transforms import recode
from src.core.preprocessing.qc import quality_control
from src.core.preprocessing.missing import handle_missing
from src.core.features import get_roi_columns_from_config

merged_all = load_and_merge(env)
baseline_df, _ = timepoint_split(env, merged_all)
baseline_df = recode(env, baseline_df)
qc_df, qc_mask = quality_control(env, baseline_df, copy=True)
clean_df = handle_missing(env, qc_df, drop_rows=True)
clean_df = clean_df[clean_df['qc_pass']].copy()

target_col = env.configs.regression['targets'][0]['column']

# Get structural ROI columns for delta brain computation
env_struct = deepcopy(env)
env_struct.configs.data['roi_features']['dopamine_core']['connectivity'] = []
struct_cols = get_roi_columns_from_config(env_struct.configs.data, ['dopamine_core'])
print(f'Structural ROI columns: {len(struct_cols)}')

# Build wide dataset with DELTA IMAGING columns
wide_df = merge_longitudinal(
    env, clean_df, merged_all, target_col, followup='year2',
    imaging_cols=struct_cols,
)

delta_col = f'{target_col}_delta'
bl_col = f'{target_col}_baseline'
y2_col = f'{target_col}_year2'

# Apply baseline severity cutoff (high-severity cohort, PPS >= 30)
bl_severity_cutoff = env.configs.regression.get('baseline_severity_cutoff', 30)
n_before = len(wide_df)
wide_df = wide_df[wide_df[bl_col] >= bl_severity_cutoff].reset_index(drop=True)
print(f'Baseline severity filter (>= {bl_severity_cutoff}): {n_before} -> {len(wide_df)} subjects')

# Build delta feature column names
delta_feature_cols = [f'{c}_delta' for c in struct_cols]
print(f'\nDelta feature columns: {len(delta_feature_cols)}')
print(f'Longitudinal dataset: {len(wide_df)} subjects')
print(f'Delta PPS: mean={wide_df[delta_col].mean():.2f}, std={wide_df[delta_col].std():.2f}')

## 2. Configure delta brain environments

Delta brain features are the predictors. ICV correction is disabled (ratio doesn't apply to difference scores).

In [ ]:
# --- Delta structural-only environment ---
env_delta = deepcopy(env)
# Point ROI features to the delta column names
env_delta.configs.data['roi_features']['dopamine_core']['structural'] = delta_feature_cols
env_delta.configs.data['roi_features']['dopamine_core']['connectivity'] = []
# Disable ICV correction for delta features
env_delta.configs.regression['icv_correction']['enabled'] = False
env_delta.configs.regression['roi_networks'] = ['dopamine_core']

print(f'Delta structural features: {len(delta_feature_cols)}')
for col in delta_feature_cols:
    n_valid = wide_df[col].notna().sum()
    print(f'  {col}: {n_valid} valid ({n_valid/len(wide_df)*100:.1f}%)')

# --- Delta combined (structural + DTI) ---
# Get connectivity columns for delta
env_combined_base = deepcopy(env)
conn_cols = get_roi_columns_from_config(env.configs.data, ['dopamine_core'])
conn_cols = [c for c in conn_cols if c not in struct_cols]  # just DTI
print(f'\nConnectivity columns for delta: {len(conn_cols)}')

# Build wide dataset with combined delta features
wide_df_combined = merge_longitudinal(
    env, clean_df, merged_all, target_col, followup='year2',
    imaging_cols=struct_cols + conn_cols,
)
# Apply baseline severity cutoff to combined dataset
n_before_comb = len(wide_df_combined)
wide_df_combined = wide_df_combined[wide_df_combined[bl_col] >= bl_severity_cutoff].reset_index(drop=True)
print(f'Baseline severity filter (combined): {n_before_comb} -> {len(wide_df_combined)} subjects')

delta_conn_cols = [f'{c}_delta' for c in conn_cols]
delta_combined_cols = delta_feature_cols + delta_conn_cols

# Filter for dMRI QC
dmri_col = 'mr_y_qc__incl__dmri_indicator'
combined_wide_df = wide_df_combined[wide_df_combined[dmri_col].astype(str) == '1'].copy().reset_index(drop=True)

env_delta_combined = deepcopy(env)
env_delta_combined.configs.data['roi_features']['dopamine_core']['structural'] = delta_feature_cols
env_delta_combined.configs.data['roi_features']['dopamine_core']['connectivity'] = delta_conn_cols
env_delta_combined.configs.regression['icv_correction']['enabled'] = False

print(f'\nDelta structural-only: {len(delta_feature_cols)} features, N={len(wide_df)}')
print(f'Delta combined: {len(delta_combined_cols)} features, N={len(combined_wide_df)}')

## 3. Delta distribution

In [ ]:
delta = wide_df[delta_col]

n_improved = (delta < 0).sum()
n_same = (delta == 0).sum()
n_worsened = (delta > 0).sum()

print(f'Improved (delta < 0): {n_improved} ({n_improved/len(delta)*100:.1f}%)')
print(f'No change (delta = 0): {n_same} ({n_same/len(delta)*100:.1f}%)')
print(f'Worsened (delta > 0): {n_worsened} ({n_worsened/len(delta)*100:.1f}%)')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(delta, bins=50, edgecolor='black', alpha=0.7, color='orange')
ax.axvline(0, color='black', linestyle='--')
ax.set_xlabel('Delta PQ-BC (Year 2 - Baseline)')
ax.set_ylabel('Count')
ax.set_title(f'Change in PQ-BC Severity (n={len(delta)})')
plt.tight_layout()
plt.show()

## 4a. Delta structural -> Delta PPS (SVR)

In [ ]:
from src.core.regression.pipeline import run_target_with_nested_cv

target_config_q3 = {'name': 'pps_severity_delta', 'column': delta_col}

# Delta structural -> Delta PPS
results_struct_q3 = run_target_with_nested_cv(
    env_delta, wide_df, target_config_q3, model_name='svr', verbose=True
)
r_struct_q3 = results_struct_q3['svr']['overall']['pearson_r']
n_struct_q3 = results_struct_q3['svr']['n_samples']
print(f'\nDelta structural -> Delta PPS: r = {r_struct_q3:.4f}, n = {n_struct_q3}')

In [ ]:
# Actual vs Predicted scatter — delta structural SVR
from scipy.stats import pearsonr
from scipy import stats as sp_stats

if 'y_true' in results_struct_q3['svr']:
    y_true_q3 = results_struct_q3['svr']['y_true']
    y_pred_q3 = results_struct_q3['svr']['y_pred']
else:
    import pickle
    seed = env_delta.configs.run['seed']
    reg_dir = (
        env_delta.repo_root / 'outputs' / env_delta.configs.run['run_name']
        / env_delta.configs.run['run_id'] / f'seed_{seed}' / 'regression'
        / target_config_q3['name'] / 'svr'
    )
    with open(reg_dir / 'results.pkl', 'rb') as f:
        saved = pickle.load(f)
    folds = saved['svr_folds']
    y_true_q3 = np.concatenate([f['y_test'] for f in folds])
    y_pred_q3 = np.concatenate([f['y_pred'] for f in folds])

rv, pv = pearsonr(y_true_q3, y_pred_q3)

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_true_q3, y_pred_q3, alpha=0.4, s=30, color='darkorange', edgecolors='saddlebrown', lw=0.5)
z = np.polyfit(y_true_q3, y_pred_q3, 1); p_line = np.poly1d(z)
xs = np.sort(y_true_q3)
ax.plot(xs, p_line(xs), 'r-', lw=2.5, zorder=9)
res = y_pred_q3 - p_line(y_true_q3)
se = np.sqrt(np.sum(res**2) / (len(y_true_q3) - 2))
tcrit = sp_stats.t.ppf(0.975, len(y_true_q3) - 2)
ci = tcrit * se * np.sqrt(1/len(y_true_q3) + (xs - y_true_q3.mean())**2 / np.sum((y_true_q3 - y_true_q3.mean())**2))
ax.fill_between(xs, p_line(xs) - ci, p_line(xs) + ci, color='red', alpha=0.12)

pstr = 'p < 0.001' if pv < 0.001 else f'p = {pv:.4f}'
ax.text(0.05, 0.97, f'r = {rv:.3f} ({pstr})\nn = {len(y_true_q3)}',
        transform=ax.transAxes, fontsize=11, va='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.9), family='monospace')
ax.set_xlabel('Observed Delta PPS (residualized)', fontsize=13, fontweight='bold')
ax.set_ylabel('Predicted', fontsize=13, fontweight='bold')
ax.set_title('Delta Brain -> Delta PPS — Structural SVR (Q3)', fontsize=14, fontweight='bold')
ax.grid(alpha=0.3, ls='--')
plt.tight_layout()
plt.show()

## 4b. Delta combined -> Delta PPS (SVR)

In [ ]:
# Delta combined (structural + DTI) -> Delta PPS
results_combined_q3 = run_target_with_nested_cv(
    env_delta_combined, combined_wide_df, target_config_q3, model_name='svr', verbose=True
)
r_combined_q3 = results_combined_q3['svr']['overall']['pearson_r']
n_combined_q3 = results_combined_q3['svr']['n_samples']
print(f'\nDelta combined -> Delta PPS: r = {r_combined_q3:.4f}, n = {n_combined_q3}')
print(f'Delta structural -> Delta PPS: r = {r_struct_q3:.4f}, n = {n_struct_q3}')

## 5. Sex-stratified SVR

In [ ]:
from src.core.regression.robustness import sex_stratified_svr

print('--- Delta structural ---')
sex_q3_struct = sex_stratified_svr(env_delta, wide_df, target_config_q3, model_name='svr')

print('\n--- Delta combined ---')
sex_q3_combined = sex_stratified_svr(env_delta_combined, combined_wide_df, target_config_q3, model_name='svr')

## 6. Permutation test

In [ ]:
from src.core.regression.evaluation import permutation_test, compute_permutation_pvalue

perm_q3 = permutation_test(
    env_delta, wide_df, target_config_q3, model_name='svr',
    n_permutations=env_delta.configs.regression['permutation']['n_permutations'], verbose=True,
)
p_emp_q3 = compute_permutation_pvalue(r_struct_q3, perm_q3['null_distribution'])
print(f'\nQ3 observed r = {r_struct_q3:.4f}, permutation p = {p_emp_q3:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(perm_q3['null_distribution'], bins=50, alpha=0.7, edgecolor='black', label='Null')
ax.axvline(r_struct_q3, color='red', linewidth=2, label=f'Observed r = {r_struct_q3:.3f}')
ax.set_xlabel('Pearson r')
ax.set_ylabel('Count')
ax.set_title(f'Q3 Permutation Test: p = {p_emp_q3:.4f}')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Univariate: Delta brain features -> Delta PPS

In [ ]:
from src.core.regression.univariate import univariate_correlations

# Correlate each delta brain feature with delta PPS (with FDR correction)
delta_target = wide_df[delta_col].values
valid_mask = np.isfinite(delta_target)
for col in delta_feature_cols:
    valid_mask &= np.isfinite(wide_df[col].values)

X_delta = wide_df.loc[valid_mask, delta_feature_cols].values.astype(float)
y_delta = delta_target[valid_mask]

short_names = [c.replace('mr_y_smri__vol__aseg__', '').replace('_delta', '_d') for c in delta_feature_cols]
corr_df_q3 = univariate_correlations(X_delta, y_delta, short_names, corrections=('bonferroni', 'fdr_bh'))
print('Univariate: Delta brain features -> Delta PPS (FDR + Bonferroni corrected):')
print(corr_df_q3[['feature', 'r', 'p_raw', 'p_bonferroni', 'sig_bonferroni',
                   'p_fdr_bh', 'sig_fdr_bh', 'n']].to_string(index=False))

In [ ]:
# Univariate delta feature correlation bar chart
fig, ax = plt.subplots(figsize=(10, max(4, len(corr_df_q3) * 0.5)))
ypos = np.arange(len(corr_df_q3))
colors = ['#d62728' if sig else '#1f77b4' if p < 0.1 else '#d9d9d9'
          for sig, p in zip(corr_df_q3['sig_fdr_bh'], corr_df_q3['p_raw'])]
ax.barh(ypos, corr_df_q3['r'].values, color=colors, edgecolor='black', lw=0.4, alpha=0.85)
ax.axvline(0, color='black', lw=1.2)
ax.set_yticks(ypos)
ax.set_yticklabels(corr_df_q3['feature'].values, fontsize=10)
ax.set_xlabel('Pearson r', fontsize=12, fontweight='bold')
ax.set_title('Univariate: Delta Brain Features vs Delta PPS', fontsize=13, fontweight='bold')
for i, row in corr_df_q3.iterrows():
    stars = '***' if row['p_fdr_bh'] < 0.001 else '**' if row['p_fdr_bh'] < 0.01 else '*' if row['p_fdr_bh'] < 0.05 else ''
    offset = 0.003 if row['r'] >= 0 else -0.003
    ha = 'left' if row['r'] >= 0 else 'right'
    ax.text(row['r'] + offset, i, f'{row["r"]:.3f}{stars}', va='center', ha=ha, fontsize=9)
ax.grid(axis='x', alpha=0.3, ls='--')
plt.tight_layout()
plt.show()

## 8. Controlling for baseline severity (residualized delta target)

In [ ]:
from sklearn.linear_model import LinearRegression

bl_vals = wide_df[bl_col].values.reshape(-1, 1)
delta_vals = wide_df[delta_col].values

resid_model = LinearRegression().fit(bl_vals, delta_vals)
delta_residualized = delta_vals - resid_model.predict(bl_vals)

wide_df_resid = wide_df.copy()
resid_col = f'{target_col}_delta_resid'
wide_df_resid[resid_col] = delta_residualized

target_config_q3_resid = {'name': 'pps_severity_delta_resid', 'column': resid_col}

# Use delta brain features -> residualized delta PPS
results_q3_resid = run_target_with_nested_cv(
    env_delta, wide_df_resid, target_config_q3_resid, model_name='svr', verbose=True
)
r_q3_resid = results_q3_resid['svr']['overall']['pearson_r']
print(f'\nQ3 (baseline-residualized delta, delta brain): r = {r_q3_resid:.4f}')
print(f'Q3 (raw delta, delta brain):                    r = {r_struct_q3:.4f}')

## 9. Summary: Q3 delta brain results

In [ ]:
print(f"{'Analysis':<45} {'r':>8} {'n':>6}")
print('-' * 62)
print(f"{'Delta structural -> Delta PPS':<45} {r_struct_q3:>+8.4f} {n_struct_q3:>6}")
print(f"{'Delta combined -> Delta PPS':<45} {r_combined_q3:>+8.4f} {n_combined_q3:>6}")
print(f"{'Delta structural -> BL-resid Delta PPS':<45} {r_q3_resid:>+8.4f} {results_q3_resid['svr']['n_samples']:>6}")

## 10. Year 4 delta brain -> delta PPS

In [ ]:
# Year 4 delta brain -> delta PPS
wide_df_y4 = merge_longitudinal(
    env, clean_df, merged_all, target_col, followup='year4',
    imaging_cols=struct_cols,
)

# Apply baseline severity cutoff
bl_col_y4 = f'{target_col}_baseline'
n_before_y4 = len(wide_df_y4)
wide_df_y4 = wide_df_y4[wide_df_y4[bl_col_y4] >= bl_severity_cutoff].reset_index(drop=True)
print(f'Baseline severity filter (Y4): {n_before_y4} -> {len(wide_df_y4)} subjects')

y4_delta_col = f'{target_col}_delta'
y4_delta_features = [f'{c}_delta' for c in struct_cols]
target_config_q3_y4 = {'name': 'pps_severity_delta_y4', 'column': y4_delta_col}

# Delta structural env for Y4
env_delta_y4 = deepcopy(env)
env_delta_y4.configs.data['roi_features']['dopamine_core']['structural'] = y4_delta_features
env_delta_y4.configs.data['roi_features']['dopamine_core']['connectivity'] = []
env_delta_y4.configs.regression['icv_correction']['enabled'] = False

print(f'Year 4 delta dataset: {len(wide_df_y4)} subjects')
results_q3_y4 = run_target_with_nested_cv(
    env_delta_y4, wide_df_y4, target_config_q3_y4, model_name='svr', verbose=True
)
r_q3_y4 = results_q3_y4['svr']['overall']['pearson_r']
n_q3_y4 = results_q3_y4['svr']['n_samples']
print(f'\nDelta structural -> Delta PPS (Y4): r = {r_q3_y4:.4f}, n = {n_q3_y4}')
print(f'Delta structural -> Delta PPS (Y2): r = {r_struct_q3:.4f}, n = {n_struct_q3}')

## 11. Delta cortical thickness -> Delta PPS

In [ ]:
# Delta cortical thickness -> Delta PPS (year 2)
cort_cols = get_roi_columns_from_config(env.configs.data, ['cortical_dopamine'])
cort_present = [c for c in cort_cols if c in merged_all.columns]

if len(cort_present) >= 2:
    wide_df_cort = merge_longitudinal(
        env, clean_df, merged_all, target_col, followup='year2',
        imaging_cols=cort_present,
    )
    # Apply baseline severity cutoff
    n_before_cort = len(wide_df_cort)
    wide_df_cort = wide_df_cort[wide_df_cort[bl_col] >= bl_severity_cutoff].reset_index(drop=True)
    print(f'Baseline severity filter (cortical): {n_before_cort} -> {len(wide_df_cort)} subjects')

    delta_cort_cols = [f'{c}_delta' for c in cort_present]

    env_delta_cort = deepcopy(env)
    env_delta_cort.configs.data['roi_features']['cortical_dopamine'] = {
        'structural': delta_cort_cols,
        'connectivity': [],
    }
    env_delta_cort.configs.regression['roi_networks'] = ['cortical_dopamine']
    env_delta_cort.configs.regression['icv_correction']['enabled'] = False

    print(f'Delta cortical features: {len(delta_cort_cols)}, N={len(wide_df_cort)}')
    results_cort_q3 = run_target_with_nested_cv(
        env_delta_cort, wide_df_cort, target_config_q3, model_name='svr', verbose=True
    )
    r_cort_q3 = results_cort_q3['svr']['overall']['pearson_r']
    n_cort_q3 = results_cort_q3['svr']['n_samples']
    print(f'\nDelta cortical -> Delta PPS (Y2): r = {r_cort_q3:.4f}, n = {n_cort_q3}')
else:
    print('Cortical thickness columns not present in data')
    r_cort_q3, n_cort_q3 = np.nan, 0

## 12. Per-region Q3 (delta brain)

In [ ]:
from src.core.regression.robustness import per_region_svr

# Per-region: delta structural -> delta PPS (with FDR-corrected p-values)
print('Per-region SVR (delta brain -> delta PPS):')
region_df_q3 = per_region_svr(env_delta, wide_df, target_config_q3, model_name='svr')
print(f'\n{region_df_q3[["region", "r", "p_raw", "p_fdr", "n"]].to_string(index=False)}')

In [ ]:
# Per-region SVR bar chart (Q3 delta brain)
fig, ax = plt.subplots(figsize=(8, max(4, len(region_df_q3) * 0.6)))
ypos = np.arange(len(region_df_q3))
colors = ['#2ca02c' if p < 0.05 else '#ff7f0e' if p < 0.1 else '#d9d9d9'
          for p in region_df_q3['p_fdr']]
ax.barh(ypos, region_df_q3['r'].values, color=colors, edgecolor='black', lw=0.5, alpha=0.85)
ax.set_yticks(ypos)
ax.set_yticklabels(region_df_q3['region'].values, fontsize=11)
ax.set_xlabel('Pearson r (5-fold CV)', fontsize=12, fontweight='bold')
ax.set_title('Per-Region SVR — Delta Brain -> Delta PPS (Q3)', fontsize=13, fontweight='bold')
ax.axvline(0, color='black', lw=1)
for i, row in region_df_q3.iterrows():
    stars = '***' if row['p_fdr'] < 0.001 else '**' if row['p_fdr'] < 0.01 else '*' if row['p_fdr'] < 0.05 else ''
    ax.text(row['r'] + 0.003, i, f'r={row["r"]:.3f} {stars}', va='center', fontsize=10)
ax.grid(axis='x', alpha=0.3, ls='--')
plt.tight_layout()
plt.show()

## 13. Delta brain -> Delta anxiety

In [ ]:
# Delta brain -> Delta anxiety (year 2)
# NOTE: Still using the PPS high-severity cohort (baseline PPS >= 30) as discriminant validity check
anx_col = 'mh_p_cbcl__dsm__anx_sum'

if anx_col in merged_all.columns:
    wide_df_anx = merge_longitudinal(
        env, clean_df, merged_all, anx_col, followup='year2',
        imaging_cols=struct_cols,
    )
    # Apply baseline PPS severity cutoff (same cohort as PPS analyses)
    pps_bl_col = f'{target_col}_baseline'
    if pps_bl_col not in wide_df_anx.columns:
        # Need to merge baseline PPS for filtering
        from src.core.preprocessing.splits import timepoint_split
        bl_tp = env.configs.data['timepoints']['baseline']
        tp_col = env.configs.data['columns']['mapping']['timepoint']
        id_col = env.configs.data['columns']['mapping']['id']
        bl_pps = merged_all[merged_all[tp_col] == bl_tp][[id_col, target_col]].rename(
            columns={target_col: pps_bl_col}
        )
        wide_df_anx = wide_df_anx.merge(bl_pps, on=id_col, how='left')
    n_before_anx = len(wide_df_anx)
    wide_df_anx = wide_df_anx[wide_df_anx[pps_bl_col] >= bl_severity_cutoff].reset_index(drop=True)
    print(f'Baseline PPS severity filter (anxiety Y2): {n_before_anx} -> {len(wide_df_anx)} subjects')

    anx_delta_col = f'{anx_col}_delta'
    anx_delta_features = [f'{c}_delta' for c in struct_cols]
    anx_target_q3 = {'name': 'cbcl_anxiety_delta', 'column': anx_delta_col}

    env_delta_anx = deepcopy(env)
    env_delta_anx.configs.data['roi_features']['dopamine_core']['structural'] = anx_delta_features
    env_delta_anx.configs.data['roi_features']['dopamine_core']['connectivity'] = []
    env_delta_anx.configs.regression['icv_correction']['enabled'] = False

    print(f'Delta brain -> Delta anxiety (Y2): N={len(wide_df_anx)}')
    results_anx_q3 = run_target_with_nested_cv(
        env_delta_anx, wide_df_anx, anx_target_q3, model_name='svr', verbose=True
    )
    r_anx_q3 = results_anx_q3['svr']['overall']['pearson_r']
    n_anx_q3 = results_anx_q3['svr']['n_samples']
    print(f'\nDelta structural -> Delta anxiety (Y2): r = {r_anx_q3:.4f}, n = {n_anx_q3}')

    # Year 4
    wide_df_anx_y4 = merge_longitudinal(
        env, clean_df, merged_all, anx_col, followup='year4',
        imaging_cols=struct_cols,
    )
    if pps_bl_col not in wide_df_anx_y4.columns:
        wide_df_anx_y4 = wide_df_anx_y4.merge(bl_pps, on=id_col, how='left')
    n_before_anx_y4 = len(wide_df_anx_y4)
    wide_df_anx_y4 = wide_df_anx_y4[wide_df_anx_y4[pps_bl_col] >= bl_severity_cutoff].reset_index(drop=True)
    print(f'Baseline PPS severity filter (anxiety Y4): {n_before_anx_y4} -> {len(wide_df_anx_y4)} subjects')

    anx_y4_delta_col = f'{anx_col}_delta'
    anx_target_q3_y4 = {'name': 'cbcl_anxiety_delta_y4', 'column': anx_y4_delta_col}

    print(f'\nDelta brain -> Delta anxiety (Y4): N={len(wide_df_anx_y4)}')
    results_anx_q3_y4 = run_target_with_nested_cv(
        env_delta_anx, wide_df_anx_y4, anx_target_q3_y4, model_name='svr', verbose=True
    )
    r_anx_q3_y4 = results_anx_q3_y4['svr']['overall']['pearson_r']
    n_anx_q3_y4 = results_anx_q3_y4['svr']['n_samples']
    print(f'Delta structural -> Delta anxiety (Y4): r = {r_anx_q3_y4:.4f}, n = {n_anx_q3_y4}')
else:
    print(f'WARNING: {anx_col} not in data')
    r_anx_q3, n_anx_q3, r_anx_q3_y4, n_anx_q3_y4 = np.nan, 0, np.nan, 0

## 14. Full Q3 summary

In [ ]:
# Full Q3 summary table
summary_rows = [
    {'Features': 'Delta subcortical (12)', 'Target': 'Delta PPS', 'Timepoint': 'Y2', 'r': r_struct_q3, 'n': n_struct_q3},
    {'Features': 'Delta combined (14)', 'Target': 'Delta PPS', 'Timepoint': 'Y2', 'r': r_combined_q3, 'n': n_combined_q3},
    {'Features': 'Delta subcortical (12)', 'Target': 'Delta PPS', 'Timepoint': 'Y4', 'r': r_q3_y4, 'n': n_q3_y4},
    {'Features': 'Delta cortical (20)', 'Target': 'Delta PPS', 'Timepoint': 'Y2', 'r': r_cort_q3, 'n': n_cort_q3},
    {'Features': 'Delta subcortical (12)', 'Target': 'Delta Anxiety', 'Timepoint': 'Y2', 'r': r_anx_q3, 'n': n_anx_q3},
    {'Features': 'Delta subcortical (12)', 'Target': 'Delta Anxiety', 'Timepoint': 'Y4', 'r': r_anx_q3_y4, 'n': n_anx_q3_y4},
    {'Features': 'Delta subcortical (12)', 'Target': 'Delta PPS (BL-resid)', 'Timepoint': 'Y2', 'r': r_q3_resid, 'n': results_q3_resid['svr']['n_samples']},
]

summary_df = pd.DataFrame(summary_rows)
summary_df['r'] = summary_df['r'].map(lambda x: f'{x:+.4f}' if pd.notna(x) else 'N/A')
print('Q3 Change Score Summary (Delta Brain -> Delta Target):')
print(summary_df.to_string(index=False))